# 01 - Orbit Usage

This notebook is the first notebook in the Nebula Space Toolkit learning path.

You will learn how to:
- create two-body and numerical orbits
- query state in multiple frames
- use multiple time input types
- inspect ephemeris coverage and precompute cached intervals
- set attitude laws and read quaternion output
- use convenience state accessors and construct from `SpacecraftState`


## How To Use This Notebook

1. Run each code cell from top to bottom.
2. Methods in `Orbit` accept scalar or vector times and return matching shapes.
3. Use `*_np` methods for raw NumPy outputs and non-`_np` methods for `astropy.units` quantities.
4. Frame strings such as `"gcrf"`, `"itrf"`, and `"native"` are supported.


In [ ]:
# Ensure local package import when running from this examples/ folder.
import sys
from pathlib import Path
%matplotlib widget
_cwd = Path.cwd().resolve()
_repo_root = _cwd if (_cwd / "nstk").is_dir() else _cwd.parent
if (_repo_root / "nstk").is_dir() and str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))


In [ ]:
from time import perf_counter

import numpy as np
import astropy.units as u
from astropy.time import Time

from nstk.propagation import Orbit
from nstk.time_utils import astropy_time_to_orekit_date

np.set_printoptions(precision=6, suppress=True)


## 1) Build A Two-Body Orbit

Use this when you want a fast analytical propagator for initial design and quick trade studies.


In [ ]:
epoch = Time("2026-01-01T00:00:00", scale="utc")
orbit = Orbit.from_kepler_two_body(
    epoch=epoch,
    a=26560e3,
    e=0.7,
    i=np.deg2rad(63.4),
    raan=np.deg2rad(20.0),
    argp=np.deg2rad(270.0),
    anomaly=np.deg2rad(10.0),
    anomaly_type="mean",
)

print("epoch:", orbit.epoch.isot)
print("native frame:", orbit.get_native_frame().getName())
print("initial coverage [s from epoch]:", orbit.coverage())

# In the notebook, this figure is interactive when `%matplotlib widget` is active.
orbit.plot()
orbit.plot(view="3d")

## 2) Query Position, Velocity, Acceleration, And Geodetic Outputs

This shows frame switching (`gcrf`, `itrf`) and geodetic conversion (`lat`, `lon`, `alt`).


In [ ]:
times = Time(epoch.unix + np.arange(0, 301, 60, dtype=np.float64), format="unix", scale="utc")
p_gcrf, v_gcrf, a_gcrf = orbit.get_pva(times, frame="gcrf")
p_itrf = orbit.get_p(times, frame="itrf")
lat, lon, alt = orbit.get_geodetic(times)

print("p_gcrf shape:", p_gcrf.shape)
print("v_gcrf shape:", v_gcrf.shape)
print("a_gcrf shape:", a_gcrf.shape)
print("p_itrf shape:", p_itrf.shape)
print("first geodetic sample [deg, deg, m]:", lat[0].value, lon[0].value, alt[0].value)


## 3) Scalar vs Vector Queries And `get_state`

- Scalar time input returns one vector (shape `(3,)`).
- Vector time input returns stacked vectors (shape `(N, 3)`).
- `get_state` can return one field (`p`, `v`, `a`) or grouped fields (`pv`, `pva`).


In [ ]:
dt_s = np.array([0.0, 120.0, 240.0], dtype=np.float64)

r_scalar = orbit.get_p(0.0, frame="gcrf", as_quantity=False)
r_vector = orbit.get_p(dt_s, frame="gcrf", as_quantity=False)
v_vector_q = orbit.get_v(dt_s, frame="gcrf")

state_pv_np = orbit.get_state(dt_s, frame="gcrf", fields="pv", as_quantity=False)
state_a_q = orbit.get_state(180.0, frame="gcrf", fields="a", as_quantity=True)

print("scalar position shape:", r_scalar.shape)
print("vector position shape:", r_vector.shape)
print("velocity quantity unit:", v_vector_q.unit)
print("state_pv_np keys:", list(state_pv_np.keys()))
print("state_pv_np['p'] shape:", state_pv_np["p"].shape)
print("state_a_q['a']:", state_a_q["a"])


## 4) Time Input Options

`Orbit` methods accept:
- `astropy.time.Time`
- seconds from epoch (float/array)
- time `Quantity` (for example `seconds * u.s`)
- Orekit `AbsoluteDate` values


In [ ]:
t_astropy = Time(epoch.unix + np.array([0.0, 30.0, 60.0]), format="unix", scale="utc")
t_seconds = np.array([0.0, 30.0, 60.0], dtype=np.float64)
t_quantity = t_seconds * u.s
t0 = astropy_time_to_orekit_date(epoch)
t_absolute = [t0.shiftedBy(0.0), t0.shiftedBy(30.0), t0.shiftedBy(60.0)]

r_astropy = orbit.get_p(t_astropy, frame="gcrf", as_quantity=False)
r_seconds = orbit.get_p(t_seconds, frame="gcrf", as_quantity=False)
r_quantity = orbit.get_p(t_quantity, frame="gcrf", as_quantity=False)
r_absolute = orbit.get_p(t_absolute, frame="gcrf", as_quantity=False)

print("allclose(astropy, seconds):", np.allclose(r_astropy, r_seconds))
print("allclose(seconds, quantity):", np.allclose(r_seconds, r_quantity))
print("allclose(seconds, AbsoluteDate):", np.allclose(r_seconds, r_absolute))


## 5) Ephemeris Precompute And Coverage

`coverage()` reports the cached ephemeris window in seconds from `epoch`, not Earth-access coverage.
That makes it a good way to see when a query forced new states to be propagated and cached.

The example below shows three things:
- a sparse query expands the cache only far enough to answer that request
- repeating the same query reuses the cached states without expanding coverage again
- `precompute(...)` fills the whole window up front before a dense query or downstream coverage workflow

Timing numbers will vary by machine and JVM warmup, but the cache-window prints should change in a predictable way.


In [ ]:
def _make_cache_demo_orbit():
    return Orbit.from_kepler_numerical(
        epoch=epoch,
        a=26560e3,
        e=0.7,
        i=np.deg2rad(63.4),
        raan=np.deg2rad(20.0),
        argp=np.deg2rad(270.0),
        anomaly=np.deg2rad(10.0),
        anomaly_type="mean",
    )

probe_dt = np.array([0.0, 6.0 * 3600.0, 12.0 * 3600.0], dtype=np.float64)
dense_dt = np.arange(0.0, 12.0 * 3600.0 + 60.0, 60.0, dtype=np.float64)

orbit_probe = _make_cache_demo_orbit()
print("fresh cache window:", orbit_probe.coverage())
_ = orbit_probe.get_p(probe_dt, frame="gcrf", as_quantity=False)
print("after sparse probe:", orbit_probe.coverage())
_ = orbit_probe.get_p(probe_dt, frame="gcrf", as_quantity=False)
print("after same probe again:", orbit_probe.coverage())

orbit_lazy = _make_cache_demo_orbit()
t0 = perf_counter()
lazy_p, lazy_v = orbit_lazy.get_pv(dense_dt, frame="gcrf", as_quantity=False)
t1 = perf_counter()
_ = orbit_lazy.get_pv(dense_dt, frame="gcrf", as_quantity=False)
t2 = perf_counter()

orbit_pre = _make_cache_demo_orbit()
print("fresh precompute orbit cache:", orbit_pre.coverage())
tp0 = perf_counter()
orbit_pre.precompute(dense_dt[0], dense_dt[-1])
tp1 = perf_counter()
print("after precompute:", orbit_pre.coverage())
pre_p, pre_v = orbit_pre.get_pv(dense_dt, frame="gcrf", as_quantity=False)
tp2 = perf_counter()

print(f"lazy first dense query  ({dense_dt.size} samples): {t1 - t0:.3f} s")
print(f"lazy second dense query ({dense_dt.size} samples): {t2 - t1:.3f} s")
print(f"precompute call                          : {tp1 - tp0:.3f} s")
print(f"precomputed dense query                  : {tp2 - tp1:.3f} s")
print("lazy cache after dense query:", orbit_lazy.coverage())
print("states match after precompute:", np.allclose(lazy_p, pre_p) and np.allclose(lazy_v, pre_v))


## 6) Numerical Orbit With Perturbations

Use this when you need higher-fidelity propagation with configurable force models.


In [ ]:
orbit_num = Orbit.from_kepler_numerical(
    epoch=epoch,
    a=7050e3,
    e=0.002,
    i=np.deg2rad(97.4),
    raan=np.deg2rad(5.0),
    argp=np.deg2rad(45.0),
    anomaly=np.deg2rad(0.0),
    anomaly_type="mean",
    gravity_degree=12,
    gravity_order=12,
    enable_third_body=True,
    third_bodies=("sun", "moon"),
    enable_srp=True,
    srp_area_m2=1.0,
    srp_cr=1.2,
    enable_drag=False,
)

state_num_pv = orbit_num.get_state(
    np.array([0.0, 120.0, 240.0], dtype=np.float64),
    frame="gcrf",
    fields="pv",
    as_quantity=False,
)
state_num_pva_q = orbit_num.get_state(300.0, frame="gcrf", fields="pva", as_quantity=True)

print("numerical propagator type:", orbit_num.propagator.__class__.__name__)
print("numerical p shape:", state_num_pv["p"].shape, "v shape:", state_num_pv["v"].shape)
print("state_num_pva_q keys:", list(state_num_pva_q.keys()))
print("type(state_num_pva_q['p']):", type(state_num_pva_q["p"]).__name__)


## 7) Attitude Law Overrides And Quaternions

`set_attitude_law` supports string aliases, dict specs, callables, and provider objects.


In [ ]:
def _attitude_callable(inertial_frame, iers, simple_eop):
    del iers, simple_eop
    from org.orekit.attitudes import LofOffset  # type: ignore
    from org.orekit.frames import LOFType  # type: ignore

    return LofOffset(inertial_frame, LOFType.QSW)


from org.orekit.attitudes import LofOffset  # type: ignore
from org.orekit.frames import LOFType  # type: ignore

t_att = np.array([0.0, 60.0, 120.0], dtype=np.float64)
q_vvlh = orbit_num.get_attitude(t_att)

orbit_num.set_attitude_law("tnw")
q_tnw = orbit_num.get_attitude(t_att)

orbit_num.set_attitude_law({"type": "nadir"})
q_nadir = orbit_num.get_attitude(t_att)

orbit_num.set_attitude_law(_attitude_callable)
q_callable = orbit_num.get_attitude(t_att)

orbit_num.set_attitude_law(LofOffset(orbit_num.get_native_frame(), LOFType.QSW))
q_provider = orbit_num.get_attitude(t_att)

orbit_num.set_attitude_law("vvlh")

print("vvlh vs tnw differs:", bool(not np.allclose(q_vvlh, q_tnw)))
print("tnw vs nadir differs:", bool(not np.allclose(q_tnw, q_nadir)))
print("nadir vs callable differs:", bool(not np.allclose(q_nadir, q_callable)))
print("provider quaternion[0] [q1, q2, q3, q4]:", q_provider[0])


## 8) Construct `Orbit` From `SpacecraftState`

This is useful when a state comes from another Orekit workflow and you want the same Nebula Space Toolkit query interface.


In [ ]:
state0 = orbit_num.propagator.getInitialState()
orbit_from_state = Orbit.from_spacecraft_state(state0, attitude="vvlh")

p0_native = orbit_from_state.get_p(0.0, frame="native", as_quantity=False)
print("from_state epoch:", orbit_from_state.epoch.isot)
print("from_state native frame:", orbit_from_state.get_native_frame().getName())
print("from_state |r0| [km]:", np.linalg.norm(p0_native) / 1e3)


Next notebook: **02 - Transforms Usage**.